# Task A: private research classification training

A lightweight single-native-T1 classifier, **not** a complete detection/localization
model or a diagnostic system. Scores are uncalibrated. Real training requires
verified MRI labels, authorization, independent geometry/native-T1 evidence,
visual QC and the existing registry/schema eligibility rules. Group/CDR never
creates labels. OASIS-2 currently cannot pass that source gate.

Keep this notebook and all outputs private after use. Disable saving outputs in
Colab. Never publish scans, identifiers, manifests, clinical records or weights.
The committed notebook contains no outputs. Default execution uses synthetic data. Use mode="synthetic_gpu" for the explicit CUDA smoke and training check. After Restart session, rerun cells from the top; CODE_ROOT is recovered automatically.
See `docs/task_a_research_training.md` for the full input and checkpoint contracts.


## 1. Mount private Drive and verify the reviewed code bundle

Select a GPU runtime for verified training; synthetic uses CPU; synthetic_gpu requires CUDA. Use Python
3.12 for this dependency profile. Build the code-only ZIP using
`scripts/build_task_a_bundle.py`; retain its SHA-256 independently and upload the
ZIP to authorized private Drive. Unpublished source is loaded from that bundle,
without cloning a branch that does not yet contain the implementation.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive').resolve()
BUNDLE_PATH = DRIVE_BASE / 'reviewed_code/task-ab-code.zip'  # Set your code-only ZIP.
BUNDLE_SHA256 = ''  # Required SHA-256 recorded independently when building the ZIP.
CONFIG_PATH = None  # Later: Path to your PRIVATE JSON config; never print its contents.


In [ ]:
import hashlib
import json
import stat
import zipfile

if (len(BUNDLE_SHA256) != 64 or any(c not in '0123456789abcdef' for c in BUNDLE_SHA256)
        or not BUNDLE_PATH.resolve().is_relative_to(DRIVE_BASE)
        or not BUNDLE_PATH.is_file() or BUNDLE_PATH.stat().st_size > 20 * 1024 * 1024):
    raise RuntimeError('Configure the private reviewed code bundle and independent SHA-256.')
blob = BUNDLE_PATH.read_bytes()
if hashlib.sha256(blob).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError('Code bundle checksum mismatch.')
CODE_ROOT = Path('/content') / ('task_ab_code_' + BUNDLE_SHA256[:16])
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    entries = archive.infolist()
    names = [entry.filename for entry in entries]
    if (len(entries) > 64 or len(set(names)) != len(names)
            or sum(entry.file_size for entry in entries) > 20 * 1024 * 1024):
        raise RuntimeError('Code bundle resource or duplicate-entry check failed.')
    for entry in entries:
        name = Path(entry.filename)
        if (name.is_absolute() or '..' in name.parts or '\\' in entry.filename
                or stat.S_ISLNK(entry.external_attr >> 16) or entry.file_size > 2 * 1024 * 1024):
            raise RuntimeError('Unsafe code bundle member.')
    manifest = json.loads(archive.read('code_manifest.json'))
    if (manifest.get('code_only') is not True or manifest.get('schema_version') != 1
            or set(names) != set(manifest['files']) | {'code_manifest.json'}):
        raise RuntimeError('Code-only allowlist check failed.')
    # Persist the verified manifest too; restart cells need it on disk.
    verified_files = {**manifest['files'], 'code_manifest.json':
                      hashlib.sha256(archive.read('code_manifest.json')).hexdigest()}
    for name, expected in verified_files.items():
        content = archive.read(name)
        if hashlib.sha256(content).hexdigest() != expected:
            raise RuntimeError('Bundled source checksum mismatch.')
        target = CODE_ROOT / name
        if any(parent.is_symlink() for parent in (target, *target.parents)):
            raise RuntimeError('Code destination symlink rejected.')
        if target.exists():
            if target.read_bytes() != content:
                raise RuntimeError('Existing code snapshot differs; use a fresh runtime.')
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
print({'reviewed_code_verified': True})

STATE_PATH = Path('/content/brainagenticthon-code-state.json')
STATE_PATH.write_text(json.dumps({'code_root': str(CODE_ROOT), 'bundle_sha256': BUNDLE_SHA256,
    'manifest_sha256': hashlib.sha256(
        (CODE_ROOT / 'code_manifest.json').read_bytes()).hexdigest()}))


## 2. Install the isolated research dependency profile

Use a fresh Colab runtime. Only software packages are installed; no datasets or
pretrained weights are fetched. If imports predate installation, restart the
runtime, rerun configuration/verification, and check versions before proceeding.


In [ ]:
# ruff: noqa: E402
import hashlib
import json
import sys
from pathlib import Path

# Survives Restart session: recover and revalidate the reviewed snapshot.
state = json.loads(Path('/content/brainagenticthon-code-state.json').read_text())
CODE_ROOT = Path(state['code_root'])
manifest_blob = (CODE_ROOT / 'code_manifest.json').read_bytes()
if hashlib.sha256(manifest_blob).hexdigest() != state['manifest_sha256']:
    raise RuntimeError('Reviewed manifest changed; rerun bundle verification.')
manifest = json.loads(manifest_blob)
for name, expected in manifest['files'].items():
    target = CODE_ROOT / name
    if (Path(name).is_absolute() or '..' in Path(name).parts
            or target.is_symlink() or not target.resolve().is_relative_to(CODE_ROOT.resolve())
            or hashlib.sha256(target.read_bytes()).hexdigest() != expected):
        raise RuntimeError('Reviewed snapshot changed; rerun bundle verification.')
import subprocess

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('This pinned research environment requires Python 3.12.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r',
                str(CODE_ROOT / 'requirements-task-a.txt')], check=True)
sys.path.insert(0, str(CODE_ROOT / 'src'))

# Check this research profile; Colab may contain unrelated incompatible packages.
from importlib.metadata import version

for requirement in (CODE_ROOT / 'requirements-task-a.txt').read_text().splitlines():
    if requirement.strip() and not requirement.startswith('#'):
        package, expected = requirement.strip().split('==')
        if version(package).split('+')[0] != expected:
            raise RuntimeError('Installed research dependency differs from reviewed pin.')


In [ ]:
# ruff: noqa: E402
import hashlib
import json
import sys
from pathlib import Path

# Survives Restart session: recover and revalidate the reviewed snapshot.
state = json.loads(Path('/content/brainagenticthon-code-state.json').read_text())
CODE_ROOT = Path(state['code_root'])
manifest_blob = (CODE_ROOT / 'code_manifest.json').read_bytes()
if hashlib.sha256(manifest_blob).hexdigest() != state['manifest_sha256']:
    raise RuntimeError('Reviewed manifest changed; rerun bundle verification.')
manifest = json.loads(manifest_blob)
for name, expected in manifest['files'].items():
    target = CODE_ROOT / name
    if (Path(name).is_absolute() or '..' in Path(name).parts
            or target.is_symlink() or not target.resolve().is_relative_to(CODE_ROOT.resolve())
            or hashlib.sha256(target.read_bytes()).hexdigest() != expected):
        raise RuntimeError('Reviewed snapshot changed; rerun bundle verification.')
sys.path.insert(0, str(CODE_ROOT / 'src'))
import nibabel as nib
import numpy as np
import torch

import brainagenticthon
from brainagenticthon.detection.data import SyntheticCohort
from brainagenticthon.detection.profile import InputProfile
from brainagenticthon.detection.training import (
    TrainConfig,
    evaluate_test,
    fit,
    load_private_cohort,
    private_output_directory,
)
from brainagenticthon.preprocessing.oasis2_mri_quality import private_call, private_path

if not Path(brainagenticthon.__file__).resolve().is_relative_to(CODE_ROOT):
    raise RuntimeError('Previously imported project differs from reviewed bundle; restart runtime.')
if (torch.__version__.split('+')[0] != '2.6.0'
        or np.__version__ != '2.5.4' or nib.__version__ != '5.4.2'):
    raise RuntimeError('Runtime imports differ from pinned packages; restart the runtime.')
print({'dependencies_verified': True, 'cuda_available': torch.cuda.is_available()})


## 3. Configure private inputs and a fresh private run directory

Start with the example's synthetic mode. For verified training, prepare the
master manifest, existing RecordReview sidecar and separate TrainingEvidence
sidecar as documented. Do not change Group/CDR or infer normality. Both classes
must be present without source/site confounding, and all patient visits must
share one frozen split. Never use the test set to select a threshold or model.

Update `CONFIG_PATH` in cell 1 to a private JSON configuration copied from the
example. Keep raw images isolated. The output directory must be private Drive,
outside Git, disjoint from images and metadata. Use a new name for each run.


In [ ]:
if CONFIG_PATH is None:
    CONFIG = json.loads((CODE_ROOT / 'configs/task_a_research.example.json').read_text())
else:
    if not Path(CONFIG_PATH).resolve().is_relative_to(DRIVE_BASE):
        raise RuntimeError('Configuration must remain in authorized private Drive.')
    CONFIG = private_call(lambda: json.loads(Path(CONFIG_PATH).read_text()))
MODE = CONFIG['mode']
if MODE not in {'synthetic', 'synthetic_gpu', 'verified'}:
    raise RuntimeError('Choose synthetic, synthetic_gpu or verified mode explicitly.')
PRIVATE_ROOT = Path(CONFIG['private_drive_root']).resolve()
if not PRIVATE_ROOT.is_relative_to(DRIVE_BASE) or PRIVATE_ROOT == DRIVE_BASE:
    raise RuntimeError('Set a dedicated private Drive project root.')
OUTPUT = private_path(PRIVATE_ROOT, CONFIG['output_run'])
if 'REPLACE_WITH' in OUTPUT.name:
    raise RuntimeError('Set a new private output_run name before executing.')
if MODE == 'verified':
    if CONFIG.get('real_training_enabled') is not True:
        raise RuntimeError('Real training is disabled; approvals and explicit opt-in are required.')
    if CONFIG['training']['device'] != 'cuda' or not torch.cuda.is_available():
        raise RuntimeError('Verified notebook training requires an available GPU runtime.')
    PROFILE = InputProfile(**{**CONFIG['profile'],
                              'target_shape': tuple(CONFIG['profile']['target_shape'])})
    TRAIN = TrainConfig(**CONFIG['training'])
else:
    settings = {**CONFIG['training'],
                'device': 'cuda' if MODE == 'synthetic_gpu' else 'cpu'}
    TRAIN = TrainConfig(**settings)
    if MODE == 'synthetic_gpu':
        from brainagenticthon.detection.training import cuda_smoke_test
        print(cuda_smoke_test())
    torch.set_num_threads(1)
print({'mode': MODE, 'private_configuration_loaded': True})


## 4. Run the mandatory eligibility gate before training

Only aggregate counts print. This is research eligibility, not clinical approval.
The real-data gate rejects any ineligible row rather than silently discarding it.
Computational checks do not replace visual QC, anatomical verification or native
T1 provenance. The original clinical policy, schema and readiness audit remain
unchanged. No labels or splits are assigned, and no source files are written.


In [ ]:
if MODE in {'synthetic', 'synthetic_gpu'}:
    COHORT = SyntheticCohort()
else:
    COHORT = private_call(
        load_private_cohort,
        private_path(PRIVATE_ROOT, CONFIG['manifest']),
        private_path(PRIVATE_ROOT, CONFIG['reviews']),
        private_path(PRIVATE_ROOT, CONFIG['training_evidence']),
        private_path(PRIVATE_ROOT, CONFIG['image_root']), PROFILE,
    )
private_output_directory(OUTPUT, COHORT.source_roots)
print(COHORT.summary)


## 5. Train or resume completed epochs

Batch=1, AMP=true and workers=0 are conservative GPU starting values; accumulation
is configurable. No private identifiers, raw paths or individual predictions are
printed. Checkpoints contain tensors/optimizer/scaler/RNG state and fingerprints,
not clinical records. Keep them private. Only resume your own trusted checkpoint.
For recovery set `resume=true` and `trusted_resume=true` with the same code,
inputs, profile and settings. Only total epochs may increase. Partial epochs are
replayed; completed epochs are not duplicated. Do not resume after final testing.


In [ ]:
if type(CONFIG.get('resume')) is not bool or type(CONFIG.get('trusted_resume')) is not bool:
    raise RuntimeError('Resume settings must be explicit JSON booleans.')
RESULT = private_call(
    fit, COHORT, TRAIN, OUTPUT,
    resume=CONFIG['resume'], trusted_resume=CONFIG['trusted_resume'],
    on_epoch=lambda aggregate: print({'epoch': aggregate['epoch'],
                                     'training_loss': aggregate['training_loss'],
                                     'validation': aggregate['validation']['overall']}),
)
print({'completed_epochs': len(RESULT.history), 'synthetic_only': MODE != 'verified',
       'clinical_approval': False, 'detection_localization_implemented': False})


## 6. Explicit final patient-level test evaluation

Leave disabled until the experiment, model-selection rule and threshold are frozen.
Test evaluation uses best validation-AUROC weights and patient-mean scores. Reports
include AUROC, sensitivity, specificity, precision, recall, confusion matrix and
anonymous source/site groups. Undefined metrics remain null. Scores have not been
clinically calibrated. Independent evaluation/calibration is still required.
Never publish this used notebook or private Drive outputs; clear all outputs.


In [ ]:
if type(CONFIG.get('final_test_evaluation')) is not bool:
    raise RuntimeError('final_test_evaluation must be an explicit JSON boolean.')
if CONFIG['final_test_evaluation']:
    FINAL_METRICS = private_call(evaluate_test, RESULT, COHORT, confirm_final_evaluation=True)
    print(FINAL_METRICS)
else:
    print({'final_test_evaluation': 'pending', 'clinical_approval': False})
